In [6]:
# Cell 2
import gymnasium as gym
import numpy as np
import random
from tqdm import tqdm # For a progress bar

# Create the environment
# render_mode="ansi" allows us to print the text representation to the console
env = gym.make("Taxi-v3", render_mode="ansi")

print("Action Space:", env.action_space)
print("State Space:", env.observation_space)

# Reset to see the initial state
state, info = env.reset()
print(env.render())

Action Space: Discrete(6)
State Space: Discrete(500)
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [7]:
# Cell 3
state_space = env.observation_space.n
action_space = env.action_space.n

# Initialize Q-table with zeros
q_table = np.zeros((state_space, action_space))

# Hyperparameters
total_episodes = 2000        # Total training episodes
learning_rate = 0.9          # Alpha: How much we update our Q-value
discount_factor = 0.95       # Gamma: Importance of future rewards
epsilon = 1.0                # Exploration rate
max_epsilon = 1.0            # Start exploration probability at 100%
min_epsilon = 0.01           # Minimum exploration probability
decay_rate = 0.005           # Exponential decay rate for epsilon
max_steps = 99               # Max steps per episode to prevent infinite loops

In [8]:
# Cell 4
print("Training started...")

for episode in tqdm(range(total_episodes)):
    # Reset environment
    state, info = env.reset()
    done = False
    
    for step in range(max_steps):
        # 1. Epsilon-Greedy Strategy: Choose Action
        exp_exp_tradeoff = random.uniform(0, 1)
        
        if exp_exp_tradeoff > epsilon:
            # Exploitation: Take the best known action
            action = np.argmax(q_table[state, :])
        else:
            # Exploration: Take a random action
            action = env.action_space.sample()

        # 2. Take the action and observe the outcome
        # Note: Gymnasium returns 5 values (new_state, reward, terminated, truncated, info)
        new_state, reward, terminated, truncated, info = env.step(action)
        
        # 3. Update Q-Table using the Bellman Equation
        # Q(s,a) = Q(s,a) + lr * [R + gamma * max(Q(s',:)) - Q(s,a)]
        q_table[state, action] = q_table[state, action] + learning_rate * (
            reward + discount_factor * np.max(q_table[new_state, :]) - q_table[state, action]
        )
        
        state = new_state
        
        # Check if episode is finished
        if terminated or truncated:
            break
            
    # Reduce epsilon (reduce exploration over time)
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)

print("\nTraining finished!")
print(f"Q-Table shape: {q_table.shape}")

Training started...


100%|██████████| 2000/2000 [00:01<00:00, 1017.27it/s]


Training finished!
Q-Table shape: (500, 6)


In [9]:
# Cell 5
total_epochs, total_penalties = 0, 0
episodes_to_evaluate = 10

print("Evaluating Agent...")

for _ in range(episodes_to_evaluate):
    state, info = env.reset()
    epochs, penalties, reward = 0, 0, 0
    terminated = False
    truncated = False
    
    # We create a new env instance for human rendering if you want to see the animation
    # But here we just print the final result to keep the notebook clean
    
    while not (terminated or truncated):
        # Always select best action (Exploitation)
        action = np.argmax(q_table[state])
        state, reward, terminated, truncated, info = env.step(action)

        if reward == -10:
            penalties += 1

        epochs += 1

    total_penalties += penalties
    total_epochs += epochs

print(f"Results after {episodes_to_evaluate} episodes:")
print(f"Average timesteps per episode: {total_epochs / episodes_to_evaluate}")
print(f"Average penalties per episode: {total_penalties / episodes_to_evaluate}")

Evaluating Agent...
Results after 10 episodes:
Average timesteps per episode: 13.1
Average penalties per episode: 0.0


In [10]:
# Cell 6
import time
from IPython.display import clear_output

# Create a human-renderable environment
env = gym.make("Taxi-v3", render_mode="ansi")

state, info = env.reset()
terminated = False
truncated = False
frames = []

while not (terminated or truncated):
    action = np.argmax(q_table[state])
    state, reward, terminated, truncated, info = env.step(action)
    
    # Render the frame
    frames.append({
        'frame': env.render(),
        'state': state,
        'action': action,
        'reward': reward
    })

# Play the animation
for i, frame in enumerate(frames):
    clear_output(wait=True)
    print(frame['frame'])
    print(f"Step: {i + 1}")
    print(f"State: {frame['state']}")
    print(f"Action: {frame['action']}")
    print(f"Reward: {frame['reward']}")
    time.sleep(0.5) # Pause to make it visible

print("Success!")

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

Step: 14
State: 475
Action: 5
Reward: 20
Success!
